<a href="https://colab.research.google.com/github/g25ait1051-collab/g25ait1051-iitj.ac.in/blob/main/NLU_Assignment_1_part_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 3: HMM POS Tagger with Viterbi

In [3]:
import nltk
from collections import defaultdict
import numpy as np

# Download the Brown corpus if not already present
try:
    nltk.data.find('corpora/brown')
except LookupError:
    print("Brown corpus not found, downloading...")
    nltk.download('brown')

# --- Data Preparation (Q11/Q13 prerequisite) ---
print("\n--- Data Preparation ---")

# Get tagged sentences from the Brown corpus
# We'll convert tags to be consistent (e.g., remove suffix for detailed tags like NN-TL, NN-PL)
# Simplification: For Brown corpus, we can use the default tags.
# Let's clean up the tags a bit for simpler handling, e.g., 'JJ-NC' -> 'JJ'
def simplify_tag(tagged_word):
    word, tag = tagged_word
    # Basic cleaning for Brown corpus tags
    if '-' in tag:
        tag = tag.split('-')[0]
    if '+' in tag:
        tag = tag.split('+')[0]
    # Handle cases like '``', "''", etc. which are often punctuation
    if tag in ['``', "''", '(', ')', ',', '.']:
        return (word, tag)
    return (word, tag.upper())

sentences = nltk.corpus.brown.tagged_sents(tagset='universal') # Use universal tagset for simplicity if preferred, or default for original.

# Let's stick to the default Brown tags for this assignment as it asks for P(ti|ti-1) and P(wi|ti)
# which implies using the original tagset unless specified otherwise.
sentences = nltk.corpus.brown.tagged_sents(categories='news') # Using 'news' category for consistency, or all categories.

# Clean tags (e.g., NN-TL -> NN)
cleaned_sentences = []
for sent in sentences:
    cleaned_sentences.append([simplify_tag(word_tag) for word_tag in sent])

# Shuffle and split data into training and testing (80/20)
np.random.seed(42) # for reproducibility
np.random.shuffle(cleaned_sentences)

split_idx = int(0.8 * len(cleaned_sentences))
train_sentences = cleaned_sentences[:split_idx]
test_sentences = cleaned_sentences[split_idx:]

print(f"Total sentences: {len(cleaned_sentences)}")
print(f"Training sentences: {len(train_sentences)}")
print(f"Testing sentences: {len(test_sentences)}")

# Extract all unique words and tags from the training data
train_words = set()
train_tags = set()
for sentence in train_sentences:
    for word, tag in sentence:
        train_words.add(word.lower()) # Convert to lower for vocabulary
        train_tags.add(tag)

# Add special start/end tags
START_TAG = '<s>'
END_TAG = '</s>'
train_tags.add(START_TAG)
train_tags.add(END_TAG)

train_tags = sorted(list(train_tags))
train_words = sorted(list(train_words))

tag_to_idx = {tag: i for i, tag in enumerate(train_tags)}
idx_to_tag = {i: tag for tag, i in tag_to_idx.items()}
word_to_idx = {word: i for i, word in enumerate(train_words)}

print(f"Unique training tags (including {START_TAG}, {END_TAG}): {len(train_tags)}")
print(f"Unique training words: {len(train_words)}")

# --- Q11: Estimate Transition and Emission Probabilities ---
print("\n--- Q11: Estimating HMM Parameters ---")

# Initialize counts with add-1 smoothing consideration (start with 1)
# Transition counts: C(t_i-1, t_i)
trans_counts = defaultdict(lambda: defaultdict(lambda: 0))
# Emission counts: C(t_i, w_i)
emiss_counts = defaultdict(lambda: defaultdict(lambda: 0))
# Tag counts: C(t_i)
tag_counts = defaultdict(lambda: 0)

# Populate counts from training data
for sentence in train_sentences:
    # Add start tag for transitions
    previous_tag = START_TAG
    tag_counts[START_TAG] += 1 # Count the start tag occurrence

    for word, current_tag in sentence:
        trans_counts[previous_tag][current_tag] += 1
        emiss_counts[current_tag][word.lower()] += 1
        tag_counts[current_tag] += 1
        previous_tag = current_tag

    # Add end tag for transitions
    trans_counts[previous_tag][END_TAG] += 1
    tag_counts[END_TAG] += 1 # Count the end tag occurrence

# Add 1 smoothing for transition and emission counts
# Total number of unique tags (including START/END) for transition smoothing denominator
n_tags = len(train_tags)
# Total number of unique words for emission smoothing denominator
n_words = len(train_words)

# Calculate transition probabilities P(t_i | t_i-1)
# A[previous_tag_idx][current_tag_idx]
A = np.zeros((n_tags, n_tags))
for prev_tag_str, current_tag_dict in trans_counts.items():
    prev_tag_idx = tag_to_idx[prev_tag_str]
    # Denominator for smoothing: C(prev_tag) + V_tags (V_tags = n_tags)
    denominator = tag_counts[prev_tag_str] + n_tags # Add-1 smoothing
    for curr_tag_str in train_tags:
        curr_tag_idx = tag_to_idx[curr_tag_str]
        # Numerator: C(prev_tag, curr_tag) + 1
        numerator = trans_counts[prev_tag_str].get(curr_tag_str, 0) + 1
        A[prev_tag_idx][curr_tag_idx] = numerator / denominator

# Calculate emission probabilities P(w_i | t_i)
# B[tag_idx][word_idx]
B = np.zeros((n_tags, n_words))
for tag_str, word_dict in emiss_counts.items():
    tag_idx = tag_to_idx[tag_str]
    # Denominator for smoothing: C(tag) + V_words (V_words = n_words)
    denominator = tag_counts[tag_str] + n_words # Add-1 smoothing
    for word_str in train_words:
        word_idx = word_to_idx[word_str]
        # Numerator: C(tag, word) + 1
        numerator = emiss_counts[tag_str].get(word_str, 0) + 1
        B[tag_idx][word_idx] = numerator / denominator

print("Transition and Emission probabilities estimated with add-1 smoothing.")

# Report 5 most probable transitions from NOUN tag
noun_tag_str = 'NN' # Brown corpus uses NN for singular noun
if noun_tag_str not in tag_to_idx:
    print(f"Warning: '{noun_tag_str}' not found in training tags. Using 'NNS' if available.")
    noun_tag_str = 'NNS' if 'NNS' in tag_to_idx else list(tag_to_idx.keys())[0] # Fallback
    print(f"Using '{noun_tag_str}' for NOUN transitions.")

noun_tag_idx = tag_to_idx.get(noun_tag_str, -1)

if noun_tag_idx != -1:
    noun_transitions = []
    for i, prob in enumerate(A[noun_tag_idx]):
        noun_transitions.append((idx_to_tag[i], prob))

    noun_transitions.sort(key=lambda x: x[1], reverse=True)
    print(f"\nTop 5 transitions from {noun_tag_str} tag:")
    for tag, prob in noun_transitions[:5]:
        print(f"  {noun_tag_str} -> {tag}: {prob:.6f}")
else:
    print(f"Could not find tag '{noun_tag_str}' to report transitions from.")


# Report 5 most probable emissions from VERB tag
verb_tag_str = 'VB' # Brown corpus uses VB for base form verb
if verb_tag_str not in tag_to_idx:
    print(f"Warning: '{verb_tag_str}' not found in training tags. Using 'VBD' if available.")
    verb_tag_str = 'VBD' if 'VBD' in tag_to_idx else list(tag_to_idx.keys())[0] # Fallback
    print(f"Using '{verb_tag_str}' for VERB emissions.")

verb_tag_idx = tag_to_idx.get(verb_tag_str, -1)

if verb_tag_idx != -1:
    verb_emissions = []
    for i, prob in enumerate(B[verb_tag_idx]):
        # 'i' here is a word index, not a tag index.
        # START_TAG and END_TAG are not words and are not emitted.
        # All words in train_words are potential emissions.
        verb_emissions.append((train_words[i], prob))

    verb_emissions.sort(key=lambda x: x[1], reverse=True)
    print(f"\nTop 5 emissions from {verb_tag_str} tag:")
    for word, prob in verb_emissions[:5]:
        print(f"  {verb_tag_str} -> {word}: {prob:.6f}")
else:
    print(f"Could not find tag '{verb_tag_str}' to report emissions from.")


# --- Q12: Implement the Viterbi Algorithm ---
print("\n--- Q12: Implementing Viterbi Algorithm ---")

def viterbi(sentence_words, A, B, tag_to_idx, idx_to_tag, word_to_idx, train_words):
    """
    Implements the Viterbi algorithm to find the most likely sequence of tags.

    Args:
        sentence_words (list): List of words in the sentence.
        A (np.array): Transition probability matrix P(t_i | t_i-1).
        B (np.array): Emission probability matrix P(w_i | t_i).
        tag_to_idx (dict): Maps tag strings to their integer indices.
        idx_to_tag (dict): Maps integer indices to tag strings.
        word_to_idx (dict): Maps word strings to their integer indices.
        train_words (list): Sorted list of words in the training vocabulary.

    Returns:
        list: The most likely sequence of tags for the sentence.
    """
    n_tags = len(idx_to_tag) # Number of possible tags
    n_words_in_sent = len(sentence_words)

    # Score matrix: dp[time_step][tag_idx] = max probability of tag_idx at time_step
    dp = np.zeros((n_words_in_sent, n_tags))
    # Backpointer matrix: backpointer[time_step][tag_idx] = tag_idx from previous step
    backpointer = np.zeros((n_words_in_sent, n_tags), dtype=int)

    start_tag_idx = tag_to_idx[START_TAG]
    # For the first word, we transition from START_TAG
    first_word = sentence_words[0].lower()
    # Handle unknown words: assign a probability distribution or use a small epsilon
    # Here, we'll assign a small emission probability if word is OOV
    if first_word in word_to_idx:
        first_word_idx = word_to_idx[first_word]
    else:
        first_word_idx = -1 # Indicate OOV

    for j in range(n_tags):
        # P(current_tag | START_TAG)
        transition_prob = A[start_tag_idx][j]

        # P(first_word | current_tag)
        if first_word_idx != -1:
            emission_prob = B[j][first_word_idx]
        else:
            # For OOV words, use a small probability for all emissions from any tag
            # This is a simplification; more robust handling might use a uniform distribution
            # or a specific OOV token probability.
            emission_prob = 1.0 / n_words # Small, non-zero probability for OOV
            if B.shape[1] > 0: # Ensure B has words dimension
                emission_prob = np.sum(B[j]) / B.shape[1] if np.sum(B[j]) > 0 else 1.0 / n_words # Average emission for the tag, or default if tag never emits

        dp[0][j] = transition_prob * emission_prob
        # No backpointer for the first step as it comes from START_TAG

    # Fill DP table for subsequent words
    for i in range(1, n_words_in_sent):
        current_word = sentence_words[i].lower()
        if current_word in word_to_idx:
            current_word_idx = word_to_idx[current_word]
        else:
            current_word_idx = -1 # Indicate OOV

        for j in range(n_tags): # Current tag (t_i)
            max_prob = 0.0
            best_prev_tag_idx = 0

            # P(current_word | current_tag)
            if current_word_idx != -1:
                emission_prob = B[j][current_word_idx]
            else:
                emission_prob = 1.0 / n_words
                if B.shape[1] > 0:
                    emission_prob = np.sum(B[j]) / B.shape[1] if np.sum(B[j]) > 0 else 1.0 / n_words

            for k in range(n_tags): # Previous tag (t_i-1)
                # dp[i-1][k] * P(current_tag | previous_tag) * P(current_word | current_tag)
                prob = dp[i-1][k] * A[k][j] * emission_prob
                if prob > max_prob:
                    max_prob = prob
                    best_prev_tag_idx = k

            dp[i][j] = max_prob
            backpointer[i][j] = best_prev_tag_idx

    # Backtrack to find the best tag sequence
    best_path = [0] * n_words_in_sent

    # Find the last tag by considering transition to END_TAG
    max_final_prob = 0.0
    last_tag_idx = 0
    end_tag_idx = tag_to_idx[END_TAG]

    for j in range(n_tags):
        # Consider the probability of dp[last_word_idx][j] * P(END_TAG | j)
        final_prob = dp[n_words_in_sent - 1][j] * A[j][end_tag_idx]
        if final_prob > max_final_prob:
            max_final_prob = final_prob
            last_tag_idx = j

    best_path[n_words_in_sent - 1] = last_tag_idx

    for i in range(n_words_in_sent - 2, -1, -1):
        best_path[i] = backpointer[i+1][best_path[i+1]]

    # Convert indices to tags
    return [idx_to_tag[idx] for idx in best_path]

print("Viterbi algorithm implemented.")

# Example usage of Viterbi (optional, for testing the function)
# test_sentence_words = [word for word, tag in test_sentences[0]]
# predicted_tags = viterbi(test_sentence_words, A, B, tag_to_idx, idx_to_tag, word_to_idx, train_words)
# print(f"\nTest sentence: {' '.join(test_sentence_words)}")
# print(f"Predicted tags: {' '.join(predicted_tags)}")
# print(f"Gold tags:      {' '.join([tag for word, tag in test_sentences[0]])}")


# --- Q13: Evaluate Tagger Accuracy ---
print("\n--- Q13: Evaluating Tagger Accuracy ---")

total_tokens = 0
correct_predictions = 0

seen_words_total = 0
seen_words_correct = 0

unseen_words_total = 0
unseen_words_correct = 0

mis_tagged_examples = [] # To store examples for Q14

for i, sentence in enumerate(test_sentences):
    words = [word for word, tag in sentence]
    gold_tags = [tag for word, tag in sentence]

    predicted_tags = viterbi(words, A, B, tag_to_idx, idx_to_tag, word_to_idx, train_words)

    for j in range(len(words)): # Iterate through tokens in the sentence
        total_tokens += 1
        word_lower = words[j].lower()
        is_seen = (word_lower in train_words)

        if predicted_tags[j] == gold_tags[j]:
            correct_predictions += 1
            if is_seen:
                seen_words_correct += 1
            else:
                unseen_words_correct += 1
        else:
            # Store mis-tagged tokens for Q14 analysis
            mis_tagged_examples.append({
                'token': words[j],
                'gold_tag': gold_tags[j],
                'predicted_tag': predicted_tags[j],
                'is_oov': not is_seen
            })

        if is_seen:
            seen_words_total += 1
        else:
            unseen_words_total += 1


token_level_accuracy = correct_predictions / total_tokens if total_tokens > 0 else 0
seen_accuracy = seen_words_correct / seen_words_total if seen_words_total > 0 else 0
unseen_accuracy = unseen_words_correct / unseen_words_total if unseen_words_total > 0 else 0

print(f"Total token-level accuracy: {token_level_accuracy:.4f}")
print(f"Accuracy for seen words: {seen_accuracy:.4f}")
print(f"Accuracy for unseen (OOV) words: {unseen_accuracy:.4f}")

print("\nComment on the difference:")
print("The accuracy for seen words is significantly higher than for unseen (OOV) words. ")
print("This is expected because the model has learned transition and emission probabilities ")
print("for seen words directly from the training data. For OOV words, the emission probabilities ")
print("are typically assigned a small, uniform value (due to add-1 smoothing or a default OOV strategy). ")
print("This means the tagger relies heavily on transition probabilities for OOV words, which is ")
print("less informative than direct emission probabilities, leading to lower accuracy.")


# --- Q14: Identify and Analyze Mis-tagged Tokens ---
print("\n--- Q14: Analyzing Mis-tagged Tokens ---")

print("\n10 Mis-tagged tokens from the test output:")
error_types = defaultdict(lambda: 0)

# Sort mis-tagged examples to prioritize OOV words for analysis if desired
# mis_tagged_examples.sort(key=lambda x: x['is_oov'], reverse=True)

for i, example in enumerate(mis_tagged_examples[:10]):
    token = example['token']
    gold_tag = example['gold_tag']
    predicted_tag = example['predicted_tag']
    is_oov = example['is_oov']

    reason = ""
    if is_oov:
        reason = "OOV word"
    elif predicted_tag in ['NN', 'NNS'] and gold_tag in ['VB', 'VBD', 'VBZ'] or \
         predicted_tag in ['VB', 'VBD', 'VBZ'] and gold_tag in ['NN', 'NNS']:
        reason = "Lexical ambiguity (Noun/Verb homograph)"
    elif predicted_tag == 'ADJ' and gold_tag == 'ADV' or \
         predicted_tag == 'ADV' and gold_tag == 'ADJ':
        reason = "Ambiguity between adjective and adverb"
    else:
        reason = "Sparse training data / Weak transition or emission probability"

    error_types[reason] += 1

    print(f"  {i+1}. Token: '{token}', Gold: {gold_tag}, Predicted: {predicted_tag}, Reason: {reason}")

print("\nSummary of error types:")
most_frequent_error = ""
max_count = 0
for error, count in error_types.items():
    print(f"  - {error}: {count} occurrences")
    if count > max_count:
        max_count = count
        most_frequent_error = error

print(f"\nMost frequent error type: '{most_frequent_error}'")
print("Why it's most frequent:")
if 'OOV word' == most_frequent_error:
    print("OOV words are common in real-world text, and the model has no direct emission ")
    print("probability for them, relying solely on contextual (transition) probabilities, ")
    print("which is less accurate.")
elif 'Lexical ambiguity (Noun/Verb homograph)' == most_frequent_error:
    print("Many words can function as both nouns and verbs (e.g., 'run', 'bank', 'light'). ")
    print("Without sufficient contextual cues or more advanced features, the tagger struggles ")
    print("to disambiguate these based purely on first-order HMM transitions.")
elif 'Ambiguity between adjective and adverb' == most_frequent_error:
    print("Adjectives and adverbs can sometimes be difficult to distinguish, especially ")
    print("when an adverb is derived from an adjective (e.g., 'fast' can be both). ")
    print("The model might struggle with their subtle syntactic differences.")
else:
    print("This indicates that the model's performance is generally limited by the ")
    print("sparseness of the training data or the inherent limitations of a simple HMM ")
    print("which only considers the previous tag for transition probabilities.")



--- Data Preparation ---
Total sentences: 4623
Training sentences: 3698
Testing sentences: 925
Unique training tags (including <s>, </s>): 101
Unique training words: 11578

--- Q11: Estimating HMM Parameters ---
Transition and Emission probabilities estimated with add-1 smoothing.

Top 5 transitions from NN tag:
  NN -> IN: 0.257413
  NN -> NN: 0.131891
  NN -> .: 0.108533
  NN -> ,: 0.101612
  NN -> NNS: 0.051199

Top 5 emissions from VB tag:
  VB -> get: 0.003758
  VB -> take: 0.002874
  VB -> see: 0.002800
  VB -> go: 0.002579
  VB -> make: 0.002579

--- Q12: Implementing Viterbi Algorithm ---
Viterbi algorithm implemented.

--- Q13: Evaluating Tagger Accuracy ---
Total token-level accuracy: 0.8216
Accuracy for seen words: 0.8643
Accuracy for unseen (OOV) words: 0.3448

Comment on the difference:
The accuracy for seen words is significantly higher than for unseen (OOV) words. 
This is expected because the model has learned transition and emission probabilities 
for seen words direc

# HMM POS Tagger Assignment Report

## 1. Data Preparation and Setup

- The Brown corpus (`nltk.corpus.brown`) was used for training and testing.
- An 80/20 train-test split was applied to the cleaned sentences (tags simplified by removing suffixes).
- Special `<s>` (start) and `</s>` (end) tags were added.
- Vocabulary size: `11578` unique training words.
- Tag set size: `101` unique training tags (including start/end).

## 2. Q11: Transition and Emission Probabilities Estimation

Transition probabilities `P(t_i | t_i-1)` and emission probabilities `P(w_i | t_i)` were estimated using Maximum Likelihood Estimation (MLE) with add-1 smoothing.

### 2.1. 5 Most Probable Transitions from the NOUN (NN) Tag

- `NN` -> `IN`: 0.257413
- `NN` -> `NN`: 0.131891
- `NN` -> `.`: 0.108533
- `NN` -> `,`: 0.101612
- `NN` -> `NNS`: 0.051199

### 2.2. 5 Most Probable Emissions from the VERB (VB) Tag

- `VB` -> `get`: 0.003758
- `VB` -> `take`: 0.002874
- `VB` -> `see`: 0.002800
- `VB` -> `go`: 0.002579
- `VB` -> `make`: 0.002579

## 3. Q12: Viterbi Algorithm Implementation

The Viterbi algorithm was implemented from scratch using dynamic programming. It maintains a score matrix (best probability up to each cell) and a backpointer matrix (which previous state gave the best score). The best tag sequence is recovered by backtracking through the backpointer matrix.

## 4. Q13: Tagger Evaluation and Accuracy

The HMM POS tagger was evaluated on the test set, and token-level accuracy was reported, distinguishing between seen and unseen (OOV) words.

- **Total token-level accuracy:** 0.8216
- **Accuracy for seen words:** 0.8643
- **Accuracy for unseen (OOV) words:** 0.3448

### Comment on the Difference:

The accuracy for seen words (0.8643) is significantly higher than for unseen (OOV) words (0.3448). This is an expected outcome. For seen words, the model has direct statistical evidence (transition and emission probabilities) from the training data. For OOV words, the emission probabilities are typically very small and uniform (due to add-1 smoothing). Consequently, the tagger relies much more on the contextual information provided by the transition probabilities for OOV words, which is less informative than direct word-to-tag mappings, leading to a substantial drop in accuracy.

## 5. Q14: Mis-tagged Tokens Analysis

10 mis-tagged tokens from the test output were identified and analyzed:

1.  **Token:** 'functions', **Gold:** `NNS`, **Predicted:** `NN`, **Reason:** OOV word
2.  **Token:** 'wage', **Gold:** `NN`, **Predicted:** `CD`, **Reason:** Sparse training data / Weak transition or emission probability
3.  **Token:** 'better', **Gold:** `JJR`, **Predicted:** `JJ`, **Reason:** Sparse training data / Weak transition or emission probability
4.  **Token:** 'as', **Gold:** `IN`, **Predicted:** `CS`, **Reason:** Sparse training data / Weak transition or emission probability
5.  **Token:** 'some', **Gold:** `DTI`, **Predicted:** `TO`, **Reason:** Sparse training data / Weak transition or emission probability
6.  **Token:** 'disappointment', **Gold:** `NN`, **Predicted:** `VB`, **Reason:** Lexical ambiguity (Noun/Verb homograph)
7.  **Token:** 'leadership', **Gold:** `NN`, **Predicted:** `WPS`, **Reason:** Sparse training data / Weak transition or emission probability
8.  **Token:** 'much', **Gold:** `RB`, **Predicted:** `AP`, **Reason:** Sparse training data / Weak transition or emission probability
9.  **Token:** 'invitations', **Gold:** `NNS`, **Predicted:** `WPS`, **Reason:** Sparse training data / Weak transition or emission probability
10. **Token:** 'Molly's', **Gold:** `NP$`, **Predicted:** `AT`, **Reason:** OOV word

### Summary of Error Types:

- **OOV word:** 2 occurrences
- **Sparse training data / Weak transition or emission probability:** 7 occurrences
- **Lexical ambiguity (Noun/Verb homograph):** 1 occurrence

### Most Frequent Error Type and Why:

The **most frequent error type is 'Sparse training data / Weak transition or emission probability'** (7 occurrences). This indicates that the model's performance is often limited by the insufficient coverage of certain tag sequences or word-tag combinations in the training data. A simple HMM, which only considers the previous tag for transition probabilities, can struggle with less common patterns or when the context isn't strong enough to overcome sparse observations. While OOV words also contribute to errors, the general sparseness of the training data appears to be a broader challenge for this model.